### Raster Model
This notebook contains the workflow for downscaling national capital and labor data. First, it uses the RF model to predict intensities at the pixel scale. Then it combines intensities with rasterized production data to get initial estimates of capital and labor in each pixel. Then it runs a re-scaling algorithim to ensure estimates align with sub-national (where available) and national totals. The output is a set of raster datasets contained estimates of capital (million USD) and labor (jobs) in each pixel. 

In [13]:
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import joblib

In [14]:
##### SET-UP

### Set directories 
# Get the current working directory
cd = Path.cwd().parent.parent 

# Set directories for saving
RESULTS_DIR = Path(f"{cd}/Results/Raster_model")

### Import data 
# RF models 
capital_rf = joblib.load(f'{cd}/Results/RF_models_final/capital_rf_final/model.joblib')
capital_qrf = joblib.load(f'{cd}/Results/RF_models_final/capital_qrf_final/model.joblib')

labor_rf = joblib.load(f'{cd}/Results/RF_models_final/labor_rf_final/model.joblib')
labor_qrf = joblib.load(f'{cd}/Results/RF_models_final/labor_qrf_final/model.joblib')

# Predictor data (stacked rasters stored in df)
predictors = pd.read_parquet(f'{cd}/Data/Clean/Training_data/raster_predictor_matrix_final.parquet')

# Production data (raster)
production_path = f'{cd}/Data/Clean/Production/total_production_tonnes_2020.tif'

# Sub-national data (for re-scaling)
capital_sub = pd.read_csv(f'{cd}/Data/Clean/Capital_stock/subnational_capital_stock_final.csv')
capital_sub_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_sub_capital_crosswalk.parquet')

labor_sub = pd.read_csv(f'{cd}/Data/Clean/Labor/subnational_labor_final.csv')
labor_sub_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_sub_labor_crosswalk.parquet')

# National data (for re-scaling)
capital_national = pd.read_csv(f'{cd}/Data/Clean/Capital_stock/FAO_capital_stock_adjusted.csv')
labor_national = pd.read_csv(f'{cd}/Data/Clean/Labor/ILO_ag_labor_estimate_adjusted.csv')
country_crosswalk = pd.read_parquet(f'{cd}/Data/Clean/Geographies/pixel_country_crosswalk.parquet')

#### Part 1: Predict intensities

In [15]:
##### PREDICTOR COLUMNS

capital_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_oilcrops_share_base_100_production_USD',
       'rtv_log_pulses_share_base_100_production_USD',
       'rtv_log_roots_tubers_share_base_100_production_USD',
       'rtv_log_sugar_crops_share_base_100_production_USD',
       'rtv_log_ruminants_share_base_100_production_USD']

labor_cols = ['rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_ruminants_share_base_100_production_USD']

In [ ]:
##### PREDICT INTENSITIES 

# isolate predictor columns
X_cap = predictors[capital_cols]
X_lab = predictors[labor_cols]

# run predictions for standard RF 
predictors['rtv_log_capital_intensity_USD_per_million_tonne'] = capital_rf.predict(X_cap)
predictors['rtv_log_labor_intensity_jobs_per_million_tonne'] = labor_rf.predict(X_lab)

# run predictions for QRF 
quantile_preds = capital_qrf.predict(X_cap, quantiles=[0.1, 0.9])
predictors['rtv_log_capital_intensity_p10'] = quantile_preds[:, 0]
predictors['rtv_log_capital_intensity_p90'] = quantile_preds[:, 1]

quantile_preds = labor_qrf.predict(X_lab, quantiles=[0.1, 0.9])
predictors['rtv_log_labor_intensity_p10'] = quantile_preds[:, 0]
predictors['rtv_log_labor_intensity_p90'] = quantile_preds[:, 1]

# reattach to x/y coords
cols_to_add = ['rtv_log_capital_intensity_USD_per_million_tonne', 
               'rtv_log_labor_intensity_jobs_per_million_tonne', 
               'rtv_log_capital_intensity_p10', 
               'rtv_log_capital_intensity_p90', 
               'rtv_log_labor_intensity_p10', 
               'rtv_log_labor_intensity_p90']
raster_predictions = predictors.pivot_table(index='y', columns='x', values=cols_to_add)

# save intermediate output (so don't have to re-run)

In [ ]:
output_path = RESULTS_DIR / "RF_predictions" / "raw_prediction.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)
raster_predictions.to_parquet(output_path, index='False')

In [ ]:
##### CONVERT UNITS

# re-import predictions 
# raster_predictions = pd.read_parquet(RESULTS_DIR / "RF_predictions" / "raw_prediction.parquet")

# add back country intensities 
raster_predictions = raster_predictions.merge(predictors[['x', 'y', 'log_country_capital_intensity_USD_per_million_tonne', 'log_country_labor_intensity_jobs_per_million_tonne']], on=[['x', 'y']], how='outer')

# convert pixel intensities from relative to absolute 
raster_predictions['capital_intensity_USD_per_million_tonne'] = np.expm1(raster_predictions['rtv_log_capital_intensity_USD_per_million_tonne'] + raster_predictions['log_country_capital_intensity_USD_per_million_tonne'])
raster_predictions['capital_intensity_p10'] = np.expm1(raster_predictions['rtv_log_capital_intensity_p10'] + raster_predictions['log_country_capital_intensity_USD_per_million_tonne'])
raster_predictions['capital_intensity_p90'] = np.expm1(raster_predictions['rtv_log_capital_intensity_p90'] + raster_predictions['log_country_capital_intensity_USD_per_million_tonne'])

raster_predictions['labor_intensity_jobs_per_million_tonne'] = np.expm1(raster_predictions['rtv_log_labor_intensity_jobs_per_million_tonne'] + raster_predictions['log_country_labor_intensity_jobs_per_million_tonne'])
raster_predictions['labor_intensity_p10'] = np.expm1(raster_predictions['rtv_log_labor_intensity_p10'] + raster_predictions['log_country_labor_intensity_jobs_per_million_tonne'])
raster_predictions['labor_intensity_p90'] = np.expm1(raster_predictions['rtv_log_labor_intensity_p90'] + raster_predictions['log_country_labor_intensity_jobs_per_million_tonne'])

